In [23]:
import numpy as np
import pandas as pd
import psycopg as pg
import mlflow
import os
from catboost import CatBoostClassifier
from mlxtend.feature_selection import SequentialFeatureSelector as SFS
from mlxtend.plotting import plot_sequential_feature_selection as plot_sfs
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import matplotlib.pyplot as plt

In [2]:
TABLE_NAME = "clean_users_churn"

TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000

EXPERIMENT_NAME = "Explore_search_hyperparams"
RUN_NAME = 'model_random_search'
REGISTRY_MODEL_NAME = "Applied_hyperparams"

In [3]:
configs = {"sslmode": "require", "target_session_attrs": "read-write"}
db_credits = {
    "host": os.getenv("DB_DESTINATION_HOST"),
    "port": os.getenv("DB_DESTINATION_PORT"),
    "dbname": os.getenv("DB_DESTINATION_NAME"),
    "user": os.getenv("DB_DESTINATION_USER"),
    "password": os.getenv("DB_DESTINATION_PASSWORD")
}

configs.update(db_credits)

In [4]:
with pg.connect(**configs) as conn:
    with conn.cursor() as cur:
        cur.execute(f"SELECT * FROM {TABLE_NAME}")

        data = cur.fetchall()

        columns = [col.name for col in cur.description]

In [5]:
df = pd.DataFrame(data=data, columns=columns)

In [6]:
features = ["monthly_charges", "total_charges", "senior_citizen"]
target = "target"

In [7]:
df[features]

,monthly_charges,total_charges,senior_citizen
0,19.80,202.25,0
1,20.15,20.15,0
2,59.90,3505.10,0
3,59.60,2970.30,0
4,55.30,1530.60,0
...,...,...,...
7019,69.50,1652.10,0
7020,76.00,1588.75,0
7021,93.60,3366.05,0
7022,95.65,778.10,0


In [9]:
test_size = 0.2
split_column = features


In [15]:
df.sort_values(by=["begin_date"])


,id,customer_id,begin_date,end_date,type,paperless_billing,payment_method,monthly_charges,total_charges,internet_service,...,device_protection,tech_support,streaming_tv,streaming_movies,gender,senior_citizen,partner,dependents,multiple_lines,target
3317,3429,9919-KNPOO,2013-10-01,2019-10-01,Two year,Yes,Bank transfer (automatic),104.15,7689.950000,Fiber optic,...,Yes,No,Yes,Yes,Male,1,Yes,No,Yes,1
4462,4612,2211-RMNHO,2013-10-01,2019-10-01,One year,Yes,Bank transfer (automatic),117.80,8684.800000,Fiber optic,...,Yes,Yes,Yes,Yes,Male,0,Yes,No,Yes,1
4367,4515,6178-KFNHS,2013-10-01,2019-10-01,Two year,No,Credit card (automatic),92.45,6440.250000,DSL,...,Yes,Yes,Yes,Yes,Female,1,Yes,Yes,Yes,1
6850,2009,2065-MMKGR,2013-11-01,2019-10-01,Two year,Yes,Credit card (automatic),108.60,7690.900000,Fiber optic,...,Yes,Yes,Yes,Yes,Male,0,Yes,No,Yes,1
951,989,5002-GCQFH,2013-11-01,2019-10-01,Two year,Yes,Electronic check,108.05,7532.150000,Fiber optic,...,Yes,Yes,Yes,Yes,Male,0,No,No,Yes,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
726,753,8405-IGQFX,2020-02-01,NaT,Two year,No,Mailed check,20.25,2291.023009,Fiber optic,...,No,No,No,No,Male,0,No,Yes,No,0
6442,6677,5850-BDWCY,2020-02-01,NaT,Two year,No,Mailed check,73.35,2291.023009,DSL,...,Yes,Yes,Yes,No,Female,0,Yes,Yes,Yes,0
1040,1094,9845-PEEKO,2020-02-01,NaT,Two year,No,Mailed check,25.75,2291.023009,Fiber optic,...,No,No,No,No,Male,0,Yes,Yes,Yes,0
464,491,3649-JPUGY,2020-02-01,NaT,Two year,Yes,Bank transfer (automatic),52.55,2291.023009,DSL,...,Yes,Yes,Yes,No,Female,0,Yes,Yes,No,0


In [16]:
X_train, X_test, y_train, y_test = train_test_split(df[features], df[target], test_size=test_size, shuffle=False)


In [17]:
X_train

,monthly_charges,total_charges,senior_citizen
3594,18.25,534.70,0
1485,18.40,1057.85,0
6423,18.55,689.00,0
6674,18.70,383.65,0
1113,18.70,1005.70,0
...,...,...,...
5852,94.25,6849.75,0
983,94.30,424.45,1
486,94.30,1818.30,1
3305,94.30,1948.35,0


In [18]:
print(f"Размер выборки для обучения: {X_train.shape}")
print(f"Размер выборки для теста: {X_test.shape}")

Размер выборки для обучения: (5619, 3)
Размер выборки для теста: (1405, 3)


In [19]:
loss_function = "Logloss"
task_type = 'CPU'
random_seed = 0
iterations = 300
verbose = False

In [ ]:
param_grid = {
    'iterations': [50, 100, 300, 500],
    'learning_rate': [0.01, 0.1, 0.3],
    'loss_function': ["Logloss", "MultiClass"],
    'depth': [4, 6, 8, 10],
    'min_data_in_leaf': [50, 100, 200],
}

In [21]:
model = CatBoostClassifier(random_seed=random_seed, task_type=task_type)

In [24]:
cv = GridSearchCV(estimator=model, param_grid=param_grid, cv=2, scoring="roc_auc", n_jobs=-1)

In [25]:
clf = cv.fit(X_train, y_train)

0:	learn: 0.6882585	total: 58.5ms	remaining: 527ms
1:	learn: 0.6835530	total: 59.2ms	remaining: 237ms
2:	learn: 0.6789331	total: 60.2ms	remaining: 140ms
3:	learn: 0.6743958	total: 60.9ms	remaining: 91.4ms
4:	learn: 0.6698802	total: 61.6ms	remaining: 61.6ms
5:	learn: 0.6654726	total: 62.3ms	remaining: 41.5ms
6:	learn: 0.6612372	total: 63.1ms	remaining: 27ms
7:	learn: 0.6571645	total: 63.7ms	remaining: 15.9ms
8:	learn: 0.6530374	total: 64.4ms	remaining: 7.15ms
9:	learn: 0.6489363	total: 65.1ms	remaining: 0us
0:	learn: 0.6876044	total: 49.3ms	remaining: 443ms
1:	learn: 0.6819988	total: 49.9ms	remaining: 200ms
2:	learn: 0.6768104	total: 50.5ms	remaining: 118ms
3:	learn: 0.6715827	total: 51ms	remaining: 76.6ms
4:	learn: 0.6663667	total: 51.6ms	remaining: 51.6ms
5:	learn: 0.6613484	total: 52.2ms	remaining: 34.8ms
6:	learn: 0.6562736	total: 52.8ms	remaining: 22.6ms
7:	learn: 0.6514023	total: 53.3ms	remaining: 13.3ms
8:	learn: 0.6465034	total: 53.9ms	remaining: 5.99ms
9:	learn: 0.6419050	total

In [26]:
print("Лучшие гиперпараметры:", clf.best_params_)
print("Лучший счет:", clf.best_score_)

Лучшие гиперпараметры: {'depth': 4, 'iterations': 30, 'learning_rate': 0.3, 'loss_function': 'MultiClass', 'min_data_in_leaf': 50}
Лучший счет: 0.7746747004484371


In [27]:
best_model = clf.best_estimator_

In [29]:
best_model.fit(X_train, y_train)

0:	learn: 0.5884494	total: 2.28ms	remaining: 66.2ms
1:	learn: 0.5270160	total: 4.14ms	remaining: 57.9ms
2:	learn: 0.4924421	total: 6.07ms	remaining: 54.6ms
3:	learn: 0.4669546	total: 8.04ms	remaining: 52.2ms
4:	learn: 0.4517155	total: 9.96ms	remaining: 49.8ms
5:	learn: 0.4395070	total: 11.8ms	remaining: 47.1ms
6:	learn: 0.4320826	total: 13.6ms	remaining: 44.8ms
7:	learn: 0.4269096	total: 16.2ms	remaining: 44.7ms
8:	learn: 0.4239272	total: 19.7ms	remaining: 46ms
9:	learn: 0.4219345	total: 22.4ms	remaining: 44.7ms
10:	learn: 0.4192349	total: 24.6ms	remaining: 42.6ms
11:	learn: 0.4171019	total: 27ms	remaining: 40.5ms
12:	learn: 0.4156591	total: 28.9ms	remaining: 37.8ms
13:	learn: 0.4144717	total: 30.7ms	remaining: 35.1ms
14:	learn: 0.4133922	total: 32.5ms	remaining: 32.5ms
15:	learn: 0.4126900	total: 34.2ms	remaining: 30ms
16:	learn: 0.4126764	total: 35.7ms	remaining: 27.3ms
17:	learn: 0.4120494	total: 37.4ms	remaining: 25ms
18:	learn: 0.4116030	total: 39.2ms	remaining: 22.7ms
19:	learn: 

In [30]:
test_score = best_model.score(X_test, y_test)

In [34]:
cv_results = pd.DataFrame(clf.cv_results_)

In [36]:
best_params = clf.best_params_

In [37]:
best_params

{'depth': 4,
 'iterations': 30,
 'learning_rate': 0.3,
 'loss_function': 'MultiClass',
 'min_data_in_leaf': 50}

In [38]:
model_best = CatBoostClassifier(**best_params, random_seed=random_seed, task_type=task_type)

In [53]:
prediction = best_model.predict(X_test)
probas = best_model.predict_proba(X_test)[:, 1]

In [42]:
metrics = {}

In [43]:
from sklearn.metrics import confusion_matrix, roc_auc_score, precision_score, recall_score, f1_score, log_loss

In [54]:
_, err1, _, err2 = confusion_matrix(y_test, prediction, normalize="all").ravel()
auc = roc_auc_score(y_test, probas)
precision = precision_score(y_test, prediction)
recall = recall_score(y_test, prediction)
f1 = f1_score(y_test, prediction)
logloss = log_loss(y_test, prediction)

In [55]:
metrics["err1"] = err1
metrics["err2"] = err2
metrics["auc"] = auc
metrics["precision"] = precision
metrics["recall"] = recall
metrics["f1"] = f1
metrics["logloss"] = logloss

In [57]:
metrics["mean_fit_time"] = cv_results["mean_fit_time"].mean()# среднее время обучения
metrics["std_fit_time"] =  cv_results["std_fit_time"].mean()# стандартное отклонение времени обучения
metrics["mean_test_score"] = cv_results["mean_test_score"].mean()# средний результат на тесте
metrics["std_test_score"] = cv_results["std_test_score"].mean()# стандартное отклонение результата на тесте
metrics["best_score"] = clf.best_score_

In [78]:
pip_requirements = "./requirements.txt"
signature = mlflow.models.infer_signature(X_test, prediction)
input_example = X_test[:10]

/home/mle-user/mle_projects/mlflow/.venv/lib/python3.10/site-packages/mlflow/models/signature.py:212: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  inputs = _infer_schema(model_input) if model_input is not None else None


In [65]:
os.environ["MLFLOW_S3_ENDPOINT_URL"] = "https://storage.yandexcloud.net"
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("AWS_SECRET_ACCESS_KEY")

In [66]:
mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")
mlflow.set_registry_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")

In [71]:
experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id

In [73]:
best_params

{'depth': 4,
 'iterations': 30,
 'learning_rate': 0.3,
 'loss_function': 'MultiClass',
 'min_data_in_leaf': 50}

In [79]:
with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    mlflow.log_metrics(metrics)
    mlflow.log_params(best_params)
    cv_info = mlflow.sklearn.log_model(cv, artifact_path="cv")
    model_info = mlflow.catboost.log_model(cb_model=best_model, artifact_path="models",
                                          registered_model_name=REGISTRY_MODEL_NAME,
                                          signature=signature,
                                           input_example=input_example,
                                           pip_requirements=pip_requirements)

Successfully registered model 'Applied_hyperparams'.
2026/01/22 15:28:11 INFO mlflow.tracking._model_registry.client: Waiting up to 300 seconds for model version to finish creation. Model name: Applied_hyperparams, version 1
Created version '1' of model 'Applied_hyperparams'.
